# Dimensionality Reduction: Principal Component Analysis (PCA)
### Mathematical Foundations, Comprehensive Intuition, and Practical Application on Global Socioeconomic Data

---

## 📌 Notebook Overview & Objectives
**Principal Component Analysis (PCA)** is an unsupervised, linear dimensionality reduction technique widely used in machine learning, statistics, and exploratory data analysis. 

In this notebook, we cover:
1. **Theoretical Foundations & Mathematical Intuition**: The Curse of Dimensionality, Covariance Matrix, Eigen-decomposition, Variance Maximization vs. Reconstruction Error Minimization.
2. **Environment Setup & Data Ingestion**: Flexible data loader (Google Colab Drive + Local environment fallback).
3. **Exploratory Data Analysis (EDA)**: Understanding feature distributions and correlation patterns.
4. **Why Feature Scaling is Critical**: Demonstrating how scale disparity distorts PCA projections.
5. **PCA Decomposition from Scratch & Scikit-Learn**: Eigenvalues, eigenvectors, and component variance.
6. **Determining the Optimal Number of Components**: Scree plot analysis, cumulative variance curve, and Kaiser's criterion.
7. **Feature Loadings & Component Interpretation**: Deciphering the latent socioeconomic meaning of PC1, PC2, and PC3.
8. **Low-Dimensional Visualizations**: 2D Projection with country annotations and 3D Interactive/Scatter Projections.
9. **The PCA Biplot**: Superimposing sample projections and feature vector directions.
10. **Unsupervised Socioeconomic Profiling**: Clustering countries in PCA space.
11. **Data Reconstruction & Reconstruction Error**: Quantifying information retention via inverse transformation.
12. **Summary, Key Takeaways, Strengths & Limitations**.


---
## 1. 📐 Mathematical Foundations of PCA

### 1.1 The Curse of Dimensionality
As feature dimensionality $d$ increases:
- The volume of the feature space grows exponentially ($V \propto r^d$), causing the available data to become extraordinarily sparse.
- Euclidean distances between arbitrary pairs of points converge to similar values: $\lim_{d \to \infty} \frac{\text{dist}_{\max} - \text{dist}_{\min}}{\text{dist}_{\min}} = 0$.
- Models suffer from severe overfitting, increased computational complexity, and multicollinearity.

Dimensionality reduction seeks a lower-dimensional representation $Z \in \mathbb{R}^{n \times k}$ (where $k \ll d$) that preserves the most vital geometric and statistical properties of the original dataset $X \in \mathbb{R}^{n \times d}$.

---

### 1.2 The Variance Maximization Objective
PCA seeks a sequence of **orthogonal unit vectors** $w_1, w_2, \dots, w_k \in \mathbb{R}^d$ such that the projection of the centered data along each vector has maximum variance.

For a centered data matrix $\tilde{X}$ (where each column has zero mean) with covariance matrix $\Sigma = \frac{1}{n-1}\tilde{X}^T \tilde{X}$:

The variance of the projected points along a unit direction $w$ ($\|w\| = 1$) is:
$$\text{Var}(\tilde{X}w) = \frac{1}{n-1}(\tilde{X}w)^T(\tilde{X}w) = w^T \left(\frac{1}{n-1}\tilde{X}^T \tilde{X}\right) w = w^T \Sigma w$$

To find the first principal component $w_1$, we formulate the constrained optimization problem:
$$\max_{w_1} \quad w_1^T \Sigma w_1 \quad \text{subject to} \quad w_1^T w_1 = 1$$

Using the **Lagrangian multiplier** method:
$$\mathcal{L}(w_1, \lambda_1) = w_1^T \Sigma w_1 - \lambda_1(w_1^T w_1 - 1)$$

Taking the partial derivative with respect to $w_1$ and setting it to 0:
$$\nabla_{w_1} \mathcal{L} = 2\Sigma w_1 - 2\lambda_1 w_1 = 0 \implies \mathbf{\Sigma w_1 = \lambda_1 w_1}$$

> **Key Theoretical Result**: The optimal direction $w_1$ is an **eigenvector** of the covariance matrix $\Sigma$, and the captured variance along that direction is exactly the corresponding **eigenvalue** $\lambda_1$:
> $$\text{Var}(\tilde{X}w_1) = w_1^T (\Sigma w_1) = w_1^T (\lambda_1 w_1) = \lambda_1 (w_1^T w_1) = \lambda_1$$

Subsequent components $w_2, w_3, \dots, w_k$ are the remaining eigenvectors sorted in descending order of their eigenvalues ($\lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_d$), subject to mutual orthogonality:
$$w_i^T w_j = 0 \quad \forall i \neq j$$

---

### 1.3 Equivalence to Minimum Reconstruction Error
PCA can also be derived as finding the linear subspace of dimension $k$ that minimizes the mean squared reconstruction error:
$$\min_{W_k} \frac{1}{n} \sum_{i=1}^n \|x_i - \hat{x}_i\|^2 \quad \text{where } \hat{x}_i = W_k W_k^T x_i$$
By the Pythagorean theorem in Euclidean space:
$$\text{Total Variance} = \text{Variance Captured by Subspace} + \text{Reconstruction Error}$$
Because total variance is fixed, **maximizing captured variance is mathematically identical to minimizing reconstruction error!**


In [ ]:
# ==========================================
# 2. Environment Setup & Data Ingestion
# ==========================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Styling parameters
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11
plt.rcParams['figure.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Resilient dataset loader (Supports Google Colab and Local Execution)
file_candidates = [
    'Country-data.csv',
    '/content/Country-data.csv',
    '/content/drive/MyDrive/Country-data.csv',
    '/content/drive/MyDrive/Hack-O-Week V sem/September 2026 week 11 and 12/Dataset/Country-data.csv'
]

data_path = None
for candidate in file_candidates:
    if os.path.exists(candidate):
        data_path = candidate
        break

if data_path is None:
    # If in Colab and drive is not mounted, attempt mount or prompt
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        for candidate in file_candidates:
            if os.path.exists(candidate):
                data_path = candidate
                break
    except Exception:
        pass

if data_path is None:
    raise FileNotFoundError("Could not locate 'Country-data.csv'. Please upload it to your Colab directory or local workspace.")

print(f"Loading dataset from: {data_path}")
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]} countries, {df.shape[1]} columns")
df.head()


---
## 3. 🔍 Exploratory Data Analysis & Multicollinearity Inspection

Before applying PCA, we must examine:
1. Data integrity: Missing values and feature types.
2. Correlation Matrix: PCA is most effective when features exhibit high multicollinearity because it compresses redundant information along correlated axes into orthogonal principal components.


In [ ]:
# ==========================================
# 3. Exploratory Data Analysis (EDA)
# ==========================================
print("--- Missing Values Check ---")
print(df.isnull().sum())

print("\n--- Statistical Summary ---")
display(df.describe().T)

# Separate country identifiers and numeric features
country_names = df['country'].values
feature_cols = [c for c in df.columns if c != 'country']
X_raw = df[feature_cols].copy()

# Correlation Matrix Heatmap
plt.figure(figsize=(10, 7))
corr = X_raw.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Matrix of Socioeconomic Indicators (Pre-PCA)', fontsize=14, pad=12)
plt.tight_layout()
plt.show()


---
## 4. ⚖️ Why Feature Scaling is Mandatory for PCA

> **Crucial Concept**: PCA seeks directions of **maximal variance**. 
> - If one feature is measured in dollars (e.g., `income` or `gdpp` ranging from $500 to $105,000, variance ~ $10^8$), and another is measured in units or percentages (e.g., `health` ~ 2% to 18%, variance ~ $10^1$), the unscaled PCA will allocate nearly **100% of the first component** simply to the feature with the largest numerical magnitude!
> - Standardizing via **Z-score normalization** transforms every feature to mean $\mu = 0$ and standard deviation $\sigma = 1$:
>   $$z = \frac{x - \mu}{\sigma}$$
>   This gives every feature equal prior weight before discovering underlying covariance structures.


In [ ]:
# ==========================================
# 4. Standard Scaling Demonstration
# ==========================================
from sklearn.preprocessing import StandardScaler

print("Variances before scaling:")
for col in feature_cols:
    print(f"  {col:12s}: {X_raw[col].var():15.2f}")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print("\nVariances after StandardScaler (All equal to 1.0):")
print(np.var(X_scaled, axis=0).round(4))


---
## 5. 🧮 Fitting PCA: Eigenvalues, Eigenvectors, and Mathematical Verification

We now fit a full PCA model ($k = d = 9$ components) to examine the entire eigenspectrum.
We will also compute the sample covariance matrix $\Sigma = \frac{1}{n-1}X_{scaled}^T X_{scaled}$ and compute its eigenvalues directly using NumPy's `np.linalg.eig` to empirically verify the Scikit-Learn implementation against the foundational mathematics.


In [ ]:
# ==========================================
# 5. Full PCA & Mathematical Verification
# ==========================================
from sklearn.decomposition import PCA

# Fit full PCA
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

eigenvalues = pca_full.explained_variance_
variance_ratio = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(variance_ratio)

# Mathematical verification via direct Covariance Matrix Eigen-decomposition
cov_matrix = np.cov(X_scaled, rowvar=False)
eigenvalues_manual, _ = np.linalg.eig(cov_matrix)
eigenvalues_manual = np.sort(eigenvalues_manual)[::-1]

# Tabulate variance explained
variance_df = pd.DataFrame({
    'Principal Component': [f'PC{i+1}' for i in range(len(eigenvalues))],
    'Eigenvalue (λ)': eigenvalues,
    'Manual Cov Eigenvalue': eigenvalues_manual,
    'Explained Variance (%)': variance_ratio * 100,
    'Cumulative Variance (%)': cumulative_variance * 100
})

print("Variance Breakdown by Principal Component:")
display(variance_df.round(3))


---
## 6. 📊 Selecting the Number of Components: Scree Plot & Kaiser Criterion

How many principal components should we retain? There are three standard criteria:
1. **The Scree Plot Elbow Rule**: Look for the "elbow" point where the marginal explained variance starts to level off (flatten out).
2. **Kaiser Criterion**: Retain components with **eigenvalue $\lambda \ge 1.0$**. Since standardized variables each contribute a variance of $1.0$, any component with $\lambda < 1.0$ contains less information than a single original variable.
3. **Cumulative Variance Threshold**: Retain enough components to capture at least **80% to 90%** of total variance.


In [ ]:
# ==========================================
# 6. Scree Plot & Cumulative Variance Curve
# ==========================================
fig, ax1 = plt.subplots(figsize=(10, 5))

# Plot Individual Explained Variance (Bars)
x = np.arange(1, len(eigenvalues) + 1)
color_bar = '#3498db'
color_line = '#e74c3c'

bars = ax1.bar(x, variance_ratio * 100, color=color_bar, alpha=0.6, label='Individual Variance (%)')
ax1.set_xlabel('Principal Component Index', fontsize=12)
ax1.set_ylabel('Individual Explained Variance (%)', color=color_bar, fontsize=12)
ax1.set_xticks(x)
ax1.set_xticklabels([f'PC{i}' for i in x])
ax1.tick_params(axis='y', labelcolor=color_bar)

# Plot Cumulative Explained Variance (Line) on secondary axis
ax2 = ax1.twinx()
ax2.plot(x, cumulative_variance * 100, color=color_line, marker='o', linewidth=2.2, label='Cumulative Variance (%)')
ax2.set_ylabel('Cumulative Explained Variance (%)', color=color_line, fontsize=12)
ax2.tick_params(axis='y', labelcolor=color_line)
ax2.set_ylim(30, 105)

# Reference Threshold lines (80% and 90%)
ax2.axhline(80, color='gray', linestyle='--', alpha=0.7, label='80% Threshold')
ax2.axhline(90, color='black', linestyle=':', alpha=0.7, label='90% Threshold')

plt.title('Scree Plot & Cumulative Explained Variance', fontsize=14, pad=12)
fig.tight_layout()
plt.show()

# Kaiser Criterion Check
kaiser_components = np.sum(eigenvalues >= 1.0)
print(f"Kaiser Criterion (Eigenvalue >= 1.0) retains {kaiser_components} components.")
print(f"Variance explained by first {kaiser_components} components: {cumulative_variance[kaiser_components - 1] * 100:.2f}%")
print(f"Variance explained by first 2 components (2D Projection): {cumulative_variance[1] * 100:.2f}%")


---
## 7. 🧬 Component Interpretation: Feature Loadings

The principal components are linear combinations of the original features:
$$PC_k = w_{1k} X_1 + w_{2k} X_2 + \dots + w_{dk} X_d$$

The weights $w_{jk}$ are known as **eigenvector loadings**.
- A high positive loading means the original feature moves in the same direction as the principal component.
- A high negative loading means an inverse relationship.
- By examining the loadings, we give concrete semantic meaning to the abstract mathematical axes.


In [ ]:
# ==========================================
# 7. Feature Loadings Heatmap & Interpretation
# ==========================================
# Components matrix shape: (n_components, n_features)
loadings = pd.DataFrame(
    pca_full.components_[:4, :].T,
    index=feature_cols,
    columns=['PC1 (Dev. vs Poverty)', 'PC2 (Trade Openness)', 'PC3 (Inflation/Stress)', 'PC4 (Demographic)']
)

plt.figure(figsize=(10, 6))
sns.heatmap(loadings, annot=True, fmt='.3f', cmap='vlag', center=0, cbar=True, linewidths=0.8)
plt.title('Feature Loadings Across Top 4 Principal Components', fontsize=14, pad=12)
plt.ylabel('Original Features')
plt.tight_layout()
plt.show()

print("Key Semantic Insights:")
print("1. PC1 (45.95% variance): Strong negative loadings on child_mort (-0.42) and total_fer (-0.40);")
print("   strong positive loadings on life_expec (+0.42), income (+0.40), and gdpp (+0.35).")
print("   -> PC1 represents the 'Socioeconomic Development & Quality of Life' Axis.")
print("2. PC2 (17.18% variance): Dominated by exports (+0.61) and imports (+0.67).")
print("   -> PC2 represents the 'Trade Openness & Globalization' Axis.")


---
## 8. 🗺️ 2D PCA Space Projection & Country Mapping

We now project all 167 nations onto the **2D plane** spanned by **PC1** and **PC2** (capturing **63.13%** of total dataset variance).
We highlight representative countries from different geopolitical and economic brackets to see how cleanly the mathematical projection separates the global spectrum.


In [ ]:
# ==========================================
# 8. 2D PCA Projection
# ==========================================
pca_2d = PCA(n_components=2, random_state=42)
X_pca_2d = pca_2d.fit_transform(X_scaled)

df_pca_2d = pd.DataFrame({
    'country': country_names,
    'PC1': X_pca_2d[:, 0],
    'PC2': X_pca_2d[:, 1],
    'gdpp': df['gdpp'],
    'child_mort': df['child_mort'],
    'income': df['income']
})

plt.figure(figsize=(13, 8))
scatter = plt.scatter(
    df_pca_2d['PC1'], 
    df_pca_2d['PC2'], 
    c=np.log1p(df_pca_2d['gdpp']), 
    cmap='viridis', 
    s=70, 
    alpha=0.85, 
    edgecolors='w', 
    linewidth=0.5
)
cbar = plt.colorbar(scatter)
cbar.set_label('Log(GDP per capita)', fontsize=11)

# Annotate select extreme and representative countries
countries_to_annotate = [
    'Haiti', 'Chad', 'Central African Republic', 'Sierra Leone', 'Afghanistan',  # Low PC1 (Vulnerable)
    'Luxembourg', 'Singapore', 'Malta', 'Ireland',                                # High PC2 (Trade hubs)
    'United States', 'Norway', 'Switzerland', 'Australia', 'Japan',               # High PC1 (Developed)
    'India', 'China', 'Brazil', 'Nigeria'                                         # Emerging / Transition
]

for _, row in df_pca_2d.iterrows():
    if row['country'] in countries_to_annotate:
        plt.annotate(
            row['country'],
            (row['PC1'], row['PC2']),
            xytext=(6, 4),
            textcoords='offset points',
            fontsize=9.5,
            fontweight='bold',
            color='#1a1a1a',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7, edgecolor='gray', linewidth=0.5)
        )

plt.axvline(0, color='gray', linestyle=':', alpha=0.5)
plt.axhline(0, color='gray', linestyle=':', alpha=0.5)
plt.title('2D PCA Projection of Countries (PC1 vs. PC2)', fontsize=14, pad=12)
plt.xlabel(f'PC1: Socioeconomic Development ({variance_ratio[0]*100:.1f}% var)', fontsize=12)
plt.ylabel(f'PC2: Trade Openness & Commerce ({variance_ratio[1]*100:.1f}% var)', fontsize=12)
plt.tight_layout()
plt.show()


---
## 9. 🎯 The PCA Biplot: Samples + Feature Vectors

A **Biplot** is one of the most powerful diagnostic tools in multivariate statistics. It plots:
1. **The Scores (Points)**: The country coordinates in principal component space.
2. **The Loadings (Vectors / Arrows)**: The direction and strength of each original variable projected into the principal component space.

**How to Interpret the Biplot:**
- **Arrow Length**: Proportional to the variance of the variable captured in this 2D plane.
- **Angle between two arrows**: Approximates the correlation between those two variables:
  - Acute angle ($< 90^\circ$): Positive correlation.
  - Perpendicular ($90^\circ$): Orthogonal / uncorrelated.
  - Obtuse angle ($> 90^\circ$): Negative correlation.
- **Position of a Country along an Arrow**: Indicates a high value of that specific variable for that country.


In [ ]:
# ==========================================
# 9. PCA Biplot (Samples + Feature Vectors)
# ==========================================
def plot_pca_biplot(score, coeff, labels, country_labels, annotate_top_n=10):
    fig, ax = plt.subplots(figsize=(13, 9))
    
    xs = score[:, 0]
    ys = score[:, 1]
    n = coeff.shape[0]
    
    # Scale factor for score points to fit comfortably on same scale as loading vectors
    scalex = 1.0 / (xs.max() - xs.min())
    scaley = 1.0 / (ys.max() - ys.min())
    
    # Scatter countries
    ax.scatter(xs * scalex, ys * scaley, c='#7f8c8d', alpha=0.5, s=40)
    
    # Plot feature vectors
    for i in range(n):
        ax.arrow(0, 0, coeff[i, 0], coeff[i, 1], color='#c0392b', alpha=0.9,
                 head_width=0.02, head_length=0.02, linewidth=1.5)
        ax.text(coeff[i, 0] * 1.15, coeff[i, 1] * 1.15, labels[i],
                color='#962d22', ha='center', va='center', fontsize=11, fontweight='bold')
        
    # Annotate select countries for context
    select = ['Haiti', 'Singapore', 'Luxembourg', 'Chad', 'United States', 'Norway', 'India', 'China']
    for idx, cname in enumerate(country_labels):
        if cname in select:
            ax.annotate(cname, (xs[idx] * scalex, ys[idx] * scaley),
                        xytext=(5, 3), textcoords='offset points', fontsize=9, fontweight='semibold')
            
    ax.axhline(0, color='grey', linestyle='--', linewidth=0.8, alpha=0.6)
    ax.axvline(0, color='grey', linestyle='--', linewidth=0.8, alpha=0.6)
    ax.set_title('PCA Biplot: Socioeconomic Indicators & Global Projections', fontsize=14, pad=12)
    ax.set_xlabel(f'Principal Component 1 ({variance_ratio[0]*100:.1f}%)', fontsize=12)
    ax.set_ylabel(f'Principal Component 2 ({variance_ratio[1]*100:.1f}%)', fontsize=12)
    plt.tight_layout()
    plt.show()

plot_pca_biplot(X_pca_2d, pca_2d.components_.T, feature_cols, country_names)


---
## 10. 🌐 3D PCA Projection

By retaining the first 3 principal components, we capture **76.14%** of the total variance across all 9 original variables.
Let us examine the 3-dimensional manifold of nations.


In [ ]:
# ==========================================
# 10. 3D PCA Projection
# ==========================================
from mpl_toolkits.mplot3d import Axes3D

pca_3d = PCA(n_components=3, random_state=42)
X_pca_3d = pca_3d.fit_transform(X_scaled)

fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection='3d')

p = ax.scatter(
    X_pca_3d[:, 0], 
    X_pca_3d[:, 1], 
    X_pca_3d[:, 2], 
    c=df['child_mort'], 
    cmap='coolwarm_r', 
    s=55, 
    edgecolors='k', 
    linewidth=0.3,
    alpha=0.85
)

cbar = fig.colorbar(p, ax=ax, pad=0.1, shrink=0.7)
cbar.set_label('Child Mortality (per 1000 live births)', fontsize=11)

ax.set_xlabel(f'PC1: Development ({variance_ratio[0]*100:.1f}%)', fontsize=10, labelpad=10)
ax.set_ylabel(f'PC2: Trade ({variance_ratio[1]*100:.1f}%)', fontsize=10, labelpad=10)
ax.set_zlabel(f'PC3: Monetary ({variance_ratio[2]*100:.1f}%)', fontsize=10, labelpad=10)
ax.set_title('3D PCA Projection (Top 3 Components = 76.1% Variance)', fontsize=14, pad=15)

plt.tight_layout()
plt.show()


---
## 11. 🏷️ Unsupervised Socioeconomic Profiling: Clustering in PCA Space

A classic real-world application of PCA is **dimensionality reduction prior to clustering**.
- Eliminates multicollinearity and noise.
- Drastically speeds up distance calculations.
- Overcomes the distance dilution effects of the curse of dimensionality.

Let us apply K-Means clustering ($k=3$) directly within the reduced PCA space and profile the resulting country clusters.


In [ ]:
# ==========================================
# 11. Clustering in Reduced PCA Space
# ==========================================
from sklearn.cluster import KMeans

# 3 Clusters corresponding to Development Tiers
kmeans = KMeans(n_clusters=3, random_state=42, n_init=15)
clusters = kmeans.fit_predict(X_pca_2d)

df_pca_2d['Cluster'] = clusters

# Profile clusters by computing mean values of original indicators
cluster_profile = df.groupby(df_pca_2d['Cluster'])[feature_cols].mean()

# Map cluster IDs to descriptive labels based on average GDP per capita
sorted_clusters_by_gdp = cluster_profile['gdpp'].sort_values().index.tolist()
label_map = {
    sorted_clusters_by_gdp[0]: 'Underdeveloped / Critical Need',
    sorted_clusters_by_gdp[1]: 'Developing / Emerging Economies',
    sorted_clusters_by_gdp[2]: 'Developed / High Income'
}
df_pca_2d['Cluster_Name'] = df_pca_2d['Cluster'].map(label_map)

plt.figure(figsize=(12, 7))
palette = {'Underdeveloped / Critical Need': '#e74c3c', 
           'Developing / Emerging Economies': '#f39c12', 
           'Developed / High Income': '#27ae60'}

sns.scatterplot(
    data=df_pca_2d, 
    x='PC1', 
    y='PC2', 
    hue='Cluster_Name', 
    palette=palette, 
    s=80, 
    alpha=0.85, 
    edgecolor='k', 
    linewidth=0.4
)

# Plot Cluster Centroids in PCA space
centroids = kmeans.cluster_centers_
plt.scatter(centroids[:, 0], centroids[:, 1], marker='X', s=250, c='black', label='Cluster Centroids', edgecolors='white', linewidth=1.5)

plt.title('Country Segmentation via K-Means on PCA Space', fontsize=14, pad=12)
plt.xlabel('PC1: Socioeconomic Development', fontsize=12)
plt.ylabel('PC2: Trade Openness', fontsize=12)
plt.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()

print("--- Cluster Mean Statistics in Original Dimensions ---")
display(df.groupby(df_pca_2d['Cluster_Name'])[feature_cols].mean().round(2).T)


---
## 12. 🔄 Data Reconstruction & Reconstruction Error

PCA is a lossy compression technique. Any projection $Z_k = \tilde{X} W_k$ can be reconstructed back to the original feature space via the inverse transformation:
$$\hat{X} = Z_k W_k^T + \mu$$

The information loss is measured by the **Mean Squared Reconstruction Error (MSE)**:
$$\text{MSE}(k) = \frac{1}{n \cdot d} \sum_{i=1}^n \sum_{j=1}^d (x_{ij} - \hat{x}_{ij})^2$$

As $k$ increases from 1 to $d$, reconstruction error monotonically decreases to 0.


In [ ]:
# ==========================================
# 12. Reconstruction Error vs. Number of Components
# ==========================================
reconstruction_errors = []
component_range = range(1, len(feature_cols) + 1)

for k in component_range:
    pca_k = PCA(n_components=k)
    X_reduced = pca_k.fit_transform(X_scaled)
    X_reconstructed = pca_k.inverse_transform(X_reduced)
    mse = np.mean((X_scaled - X_reconstructed) ** 2)
    reconstruction_errors.append(mse)

plt.figure(figsize=(9, 5))
plt.plot(component_range, reconstruction_errors, marker='s', color='#8e44ad', linewidth=2.2, markersize=7)
plt.title('Reconstruction Error (MSE) vs. Number of Principal Components', fontsize=14, pad=12)
plt.xlabel('Number of Components Retained (k)', fontsize=12)
plt.ylabel('Mean Squared Reconstruction Error', fontsize=12)
plt.xticks(list(component_range))
plt.grid(True, linestyle='--', alpha=0.6)

# Annotate error at k=2, 4, 9
for k_val in [2, 4, 9]:
    err = reconstruction_errors[k_val - 1]
    plt.annotate(f'k={k_val}: MSE={err:.3f}', (k_val, err),
                 xytext=(15, 10), textcoords='offset points',
                 arrowprops=dict(arrowstyle='->', color='black'))

plt.tight_layout()
plt.show()


---
## 13. 📝 Summary, Assumptions & Limitations of PCA

### Summary of Findings on Country Data:
1. **Dimension Reduction**: 9 socioeconomic indicators were successfully compressed into 2 components retaining **63.1%** variance and 4 components retaining **87.2%** variance.
2. **Latent Factors**:
   - PC1 reflects **Quality of Life & Development** (contrasting high GDP/Life Expectancy against Child Mortality/Fertility).
   - PC2 reflects **Trade Openness & Global Integration** (Imports & Exports).
3. **Clustering Power**: PCA separated countries cleanly into actionable socioeconomic groups (Underdeveloped, Emerging, Developed).

---

### Core Assumptions of PCA:
- **Linearity**: Assumes the data manifold can be approximated by a linear subspace. If relationships are complex curves or non-linear manifolds, PCA will fail to capture the true underlying topology.
- **Orthogonality**: Assumes orthogonal axes are the most meaningful representations.
- **Variance as Information**: Assumes axes with large variance contain the most useful information. While often true for unsupervised exploration, an axis of small variance could theoretically hold the strongest predictive signal for a supervised classification task.
- **Scale Sensitivity**: Requires standard scaling; otherwise, high-magnitude features dominate arbitrarily.

---

### PCA vs. Non-Linear Dimensionality Reduction:
| Dimension | PCA | Non-Linear Methods (t-SNE, UMAP) |
|---|---|---|
| **Mapping Type** | Linear Projection | Non-Linear Manifold Embedding |
| **Preservation** | Global Variance & Distances | Local Neighborhood Topologies |
| **Deterministic?** | Yes (Exact Eigen-decomposition) | Stochastic (Iterative Gradient Optimization) |
| **Out-of-sample data** | Yes (can transform new data: `pca.transform(X_new)`) | No (t-SNE cannot embed new points without refitting) |
| **Primary Use Cases** | Compression, feature extraction, noise reduction | Exploratory cluster visualization |

*Next Step*: Proceed to **`t-SNE.ipynb`** to explore how non-linear dimensionality reduction handles complex manifold structures and overcomes the linear limits of PCA!
